In [61]:
from dataclasses import dataclass

In [62]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    raw_data_file: Path
    ticker: str
    start_date: str
    end_date: str
    

In [63]:
!pwd
import os
# os.chdir("../")

/c/Users/Ibk/Desktop/data project/Stock-Price-Prediction-MLOps


In [64]:
from src.stock_prediction.utils.common import *
read_yaml(CONFIG_FILE_PATH)

2026-08-07 10:33:06,663 | INFO | common| YAML file: config\config.yaml loaded successfully.


ConfigBox({'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'raw_data_file': 'artifacts/data_ingestion/stock_data.csv', 'ticker': 'AAPL', 'start_date': datetime.date(2000, 1, 1), 'end_date': datetime.date(2026, 8, 4)}})

In [65]:
from stock_prediction.constants import *
from src.stock_prediction.utils.common import *
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            raw_data_file=Path(config.raw_data_file),
            ticker=config.ticker,
            start_date=config.start_date,
            end_date=config.end_date
        )
        return data_ingestion_config

In [66]:
a = ConfigurationManager()
a.get_data_ingestion_config()

2026-08-07 10:33:06,702 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-07 10:33:06,713 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-07 10:33:06,716 | INFO | common| Directory created at: artifacts
2026-08-07 10:33:06,719 | INFO | common| Directory created at: artifacts/data_ingestion


DataIngestionConfig(root_dir='artifacts/data_ingestion', raw_data_file=WindowsPath('artifacts/data_ingestion/stock_data.csv'), ticker='AAPL', start_date=datetime.date(2000, 1, 1), end_date=datetime.date(2026, 8, 4))

In [67]:
import yfinance as yf
yf.download?

Signature:
yf.download(
    tickers,
    start=None,
    end=None,
    actions=False,
    threads=True,
    ignore_tz=None,
    group_by='column',
    auto_adjust=True,
    back_adjust=False,
    repair=False,
    keepna=False,
    progress=True,
    period='1mo if start & end None',
    interval='1d',
    prepost=False,
    rounding=False,
    timeout=10,
    session=None,
    multi_level_index=True,
) -> Optional[pandas.core.frame.DataFrame]
Docstring:
Download yahoo tickers
:Parameters:
    tickers : str, list
        List of tickers to download
    period : str
        Valid periods: 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max
        Default: '1mo' if start & end None
        Either Use period parameter or use start and end
    interval : str
        Valid intervals: 1m,2m,5m,15m,30m,60m,90m,1h,1d,5d,1wk,1mo,3mo
        Intraday data cannot extend last 60 days
    start: str
        Download start date string (YYYY-MM-DD) or _datetime, inclusive.
        Default is 99 years ago
       

Signature:
yf.download(
    tickers,
    start=None,
    end=None,
    actions=False,
    threads=True,
    ignore_tz=None,
    group_by='column',
    auto_adjust=True,
    back_adjust=False,
    repair=False,
    keepna=False,
    progress=True,
    period='1mo if start & end None',
    interval='1d',
    prepost=False,
    rounding=False,
    timeout=10,
    session=None,
    multi_level_index=True,
) -> Optional[pandas.core.frame.DataFrame]
Docstring:
Download yahoo tickers
:Parameters:
    tickers : str, list
        List of tickers to download
    period : str
        Valid periods: 1d,5d,1mo,3mo,6mo,1y,2y,5y,10y,ytd,max
        Default: '1mo' if start & end None
        Either Use period parameter or use start and end
    interval : str
        Valid intervals: 1m,2m,5m,15m,30m,60m,90m,1h,1d,5d,1wk,1mo,3mo
        Intraday data cannot extend last 60 days
    start: str
        Download start date string (YYYY-MM-DD) or _datetime, inclusive.
        Default is 99 years ago
       

In [68]:
# !pip install yfinance

In [69]:
from src.stock_prediction.utils.common import save_json



In [70]:
import pandas as pd
from pathlib import Path
def save_csv(data: pd.DataFrame, file_path: Path) -> None:
    """Saves the DataFrame to a CSV file.

    Args:
        data (pd.DataFrame): The DataFrame to save.
        file_path (Path): The path where the CSV file will be saved.
    """
    try:
        data.to_csv(file_path, index=False)
        logger.info(f"Data saved to {file_path}")
    except Exception as e:
        logger.error(f"Error saving data to {file_path}: {e}")
        raise e

In [71]:
import yfinance as yf
from stock_prediction import logger
import pandas as pd
from src.stock_prediction.entity.config_entity import DataIngestionConfig
from src.stock_prediction.utils.common import *

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config =  config
    def fetch_file(self):
        try:
            
            logger.info(f"Data downloading for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            data = yf.download(
                tickers=self.config.ticker,
                start=self.config.start_date,
                end=self.config.end_date
            )
            logger.info(f"Data downloaded successfully for ticker: {self.config.ticker} from \
                {self.config.start_date} to {self.config.end_date}")
            if not (data.empty):
                data = self.reset_columns(data)           
                save_csv(data, self.config.raw_data_file)

            else:
                logger.error(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}") 
                raise ValueError(f"No data found for ticker: {self.config.ticker} from \
                    {self.config.start_date} to {self.config.end_date}")
        except Exception as e:
            logger.error(f"Error while downloading data for ticker: {self.config.ticker} ")
            raise e

        
        
    def reset_columns(self, data: pd.DataFrame) -> pd.DataFrame:
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.droplevel(1)
        data = data.reset_index()
        return data

In [72]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(data_ingestion_config)
    data_ingestion.fetch_file()
except Exception as e:
    logger.exception(e)
    raise e

2026-08-07 10:33:06,999 | INFO | common| YAML file: config\config.yaml loaded successfully.
2026-08-07 10:33:07,002 | INFO | common| YAML file: params.yaml loaded successfully.
2026-08-07 10:33:07,004 | INFO | common| Directory created at: artifacts
2026-08-07 10:33:07,006 | INFO | common| Directory created at: artifacts/data_ingestion
2026-08-07 10:33:07,008 | INFO | 2975460377| Data downloading for ticker: AAPL from                 2000-01-01 to 2026-08-04


[*********************100%***********************]  1 of 1 completed

2026-08-07 10:33:07,165 | INFO | 2975460377| Data downloaded successfully for ticker: AAPL from                 2000-01-01 to 2026-08-04
2026-08-07 10:33:07,215 | INFO | common| Data saved to artifacts\data_ingestion\stock_data.csv


In [73]:
type(data_ingestion_config.start_date)

datetime.date

In [74]:
type(data.columns)

pandas.core.indexes.multi.MultiIndex

In [75]:
data.head(1)

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-01-01,100,105,95,98,1000000


In [76]:
data.columns.droplevel(1)

Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')

In [77]:
data.reset_index()

Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2026-01-01,100,105,95,98,1000000
1,2026-01-02,101,106,96,99,1100000
2,2026-01-03,102,107,97,100,1200000


In [78]:
import pandas as pd
pd.MultiIndex

pandas.core.indexes.multi.MultiIndex

In [ ]:
import pytest
import pandas as pd
import pytest
from pathlib import Path
from unittest.mock import patch, MagicMock
from stock_prediction.components.data_ingestion import DataIngestion
from stock_prediction.entity.config_entity import DataIngestionConfig



@pytest.fixture
def sample_config(tmp_path):
    data_ingestion_config =  DataIngestionConfig(
        root_dir=tmp_path,
        ticker="AAPL",
        start_date="2020-01-01",
        end_date="2020-01-10",
        raw_data_file=tmp_path / "stock_data.csv",
    )
    return data_ingestion_config

@pytest.fixture
def multiindex_dataframe():
    dates = pd.date_range("2026-01-01", periods=3)
    columns = pd.MultiIndex.from_product(
        [["Close", "High", "Low", "Open", "Volume"], ["AAPL"]],
        names=["Price", "Ticker"]
    )
    data = pd.DataFrame(
                [[100, 105, 95, 98, 1000000],
            [101, 106, 96, 99, 1100000],
            [102, 107, 97, 100, 1200000]],
                index=dates, columns=columns
    )

    data.index.name = "Date"
    return data

In [ ]:
#multiindex_dataframe()

Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2026-01-01,100,105,95,98,1000000
2026-01-02,101,106,96,99,1100000
2026-01-03,102,107,97,100,1200000


In [ ]:
class TestResetColumns:
    def test_flattens_multiindex(self, sample_config, multiindex_dataframe):
        ingestion = DataIngestion(config=sample_config)
        result = ingestion.reset_columns(multiindex_dataframe)
        assert not isinstance(result.columns, pd.MultiIndex)
        assert result.columns == ["Date", "Close", "High", "Low", "Open", "Volume"]
        
    def test_date_becomes_a_column(self, sample_config, multiindex_dataframe):
        ingestion = DataIngestion(config=sample_config)
        result = ingestion.reset_columns(multiindex_dataframe)
        assert "Date" in result.columns
        assert pd.api.types.is_datetime64_any_dtype(result["Date"])
        
    def test_handles_already_flat_columns(self, sample_config):
        flat_data = pd.DataFrame(
            {"Close": [100], "High": [105], "Low": [95], "Open": [98], "Volume": [1000000]},
            pd.date_range("2026-01-01", periods=1)
        )
        flat_data.index.name = "Date"
        ingestion = DataIngestion(config=sample_config)
        result = ingestion.reset_columns(flat_data)
        assert list(result.columns) == ["Date", "Close", "High", "Low", "Open", "Volume"]
        


In [ ]:
class TestFetchFile:
    
    @patch("stock_prediction.components.data_ingestion.yf.download")
    def test_saves_csv_on_successful_download(self, mock_download, sample_config, multiindex_dataframe):
        mock_download.return_value = multiindex_dataframe
        ingestion = DataIngestion(config=sample_config)
        ingestion.fetch_file()
        
        assert sample_config.raw_data_file.exists()
        saved = pd.read_csv(sample_config.raw_data_file)
        assert list(saved.columns) == ["Date", "Close", "High", "Low", "Open", "Volume"]
        assert len(saved) == 3
        
    @patch("stock_prediction.components.data_ingestion.yf.download")     
    def test_raises_on_empty_data(self, mock_download, sample_config):
        mock_download.return_value = pd.DataFrame()
        ingestion = DataIngestion(config=sample_config)
        with pytest.raises(ValueError, match="No data found"):
            ingestion.fetch_file()
        assert not sample_config.raw_data_file.exists()
        
    @patch("stock_prediction.components.data_ingestion.yf.download")            
    def test_raises_on_yfinance_newtork_error(self, mock_download, sample_config):
        mock_download.side_effect = ConnectionError("network unreachable")
        ingestion = DataIngestion(config=sample_config)
        with pytest.raises(ConnectionError):
            ingestion.fetch_file()

In [ ]:



class TestFetchFile:

    @patch("stock_prediction.components.data_ingestion.yf.download")
    def test_raises_on_empty_data(self, mock_download, sample_config):
        mock_download.return_value = pd.DataFrame()

        ingestion = DataIngestion(config=sample_config)

        with pytest.raises(ValueError, match="No data found"):
            ingestion.fetch_file()

        assert not sample_config.raw_data_file.exists()

    @patch("stock_prediction.components.data_ingestion.yf.download")
    def test_does_not_save_file_when_empty(self, mock_download, sample_config):
        """Regression test for the ordering bug: empty data should
        never reach disk."""
        mock_download.return_value = pd.DataFrame()

        ingestion = DataIngestion(config=sample_config)

        with pytest.raises(ValueError):
            ingestion.fetch_file()

        assert not sample_config.raw_data_file.exists()

    @patch("stock_prediction.components.data_ingestion.yf.download")
    def test_raises_on_yfinance_network_error(self, mock_download, sample_config):
        mock_download.side_effect = ConnectionError("network unreachable")

        ingestion = DataIngestion(config=sample_config)

        with pytest.raises(ConnectionError):
            ingestion.fetch_file()